In [1]:
"""
bias_audit.py
--------------
Auditoria de sesgo (fairness) del modelo campeon del pipeline de Threat
Hunting Autonomo, sobre el propio dataset del proyecto (CSE-CIC-IDS2018).

Notebook autocontenido: no depende de variables cargadas por otro
notebook. Vuelve a montar Drive, reconstruye el modelo campeon desde su
archivo .pt guardado por train_pipeline.py, y recalcula todo lo que
necesita desde cero.

Atributo protegido / proxy usado: rango de 'Dst Port' del flujo
(well-known / registered / dynamic). 'Dst Port' se excluye del
entrenamiento del modelo (es una columna de leakage, ver
preprocessing.py), pero se reutiliza aqui SOLO como variable de
agrupacion para auditar si el modelo campeon comete mas falsos
negativos en unos grupos de trafico que en otros -- un riesgo real en
un SOC (segmentos "ciegos" al modelo).
"""

'\nbias_audit.py\n--------------\nAuditoria de sesgo (fairness) del modelo campeon del pipeline de Threat\nHunting Autonomo, sobre el propio dataset del proyecto (CSE-CIC-IDS2018).\n\nNotebook autocontenido: no depende de variables cargadas por otro\nnotebook. Vuelve a montar Drive, reconstruye el modelo campeon desde su\narchivo .pt guardado por train_pipeline.py, y recalcula todo lo que\nnecesita desde cero.\n\nAtributo protegido / proxy usado: rango de \'Dst Port\' del flujo\n(well-known / registered / dynamic). \'Dst Port\' se excluye del\nentrenamiento del modelo (es una columna de leakage, ver\npreprocessing.py), pero se reutiliza aqui SOLO como variable de\nagrupacion para auditar si el modelo campeon comete mas falsos\nnegativos en unos grupos de trafico que en otros -- un riesgo real en\nun SOC (segmentos "ciegos" al modelo).\n'

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import sys
PROJECT_DIR = "/content/drive/MyDrive/Trabajo_Cualitativo"
if PROJECT_DIR not in sys.path:
    sys.path.append(PROJECT_DIR)

# Asegura que preprocessing.py y models.py existan como modulos .py
!jupyter nbconvert --to python "$PROJECT_DIR/preprocessing.ipynb"
!jupyter nbconvert --to python "$PROJECT_DIR/models.ipynb"

[NbConvertApp] Converting notebook /content/drive/MyDrive/Trabajo_Cualitativo/preprocessing.ipynb to python
[NbConvertApp] Writing 5391 bytes to /content/drive/MyDrive/Trabajo_Cualitativo/preprocessing.py
[NbConvertApp] Converting notebook /content/drive/MyDrive/Trabajo_Cualitativo/models.ipynb to python
[NbConvertApp] Writing 3362 bytes to /content/drive/MyDrive/Trabajo_Cualitativo/models.py


In [4]:
import json
import os

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split

from preprocessing import clean, encode_target, prepare_dataset
from models import build_model

# --------------------------------------------------------------------- #
# Configuracion (mismas rutas y semillas que train_pipeline.py)
# --------------------------------------------------------------------- #
PROJECT_DIR = "/content/drive/MyDrive/Trabajo_Cualitativo"
DATA_PATH = f"{PROJECT_DIR}/Data/02-15-2018.csv"
RESULTS_DIR = f"{PROJECT_DIR}/results"
SUMMARY_JSON_PATH = f"{RESULTS_DIR}/summary_statistics.json"
ALL_RUNS_PATH = f"{RESULTS_DIR}/all_runs.csv"
BIAS_AUDIT_DIR = f"{RESULTS_DIR}/bias_audit"
os.makedirs(BIAS_AUDIT_DIR, exist_ok=True)

SPLIT_SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

PRIMARY_METRIC = "f1_mean"
STD_TIEBREAKER = "f1_std"
SECONDARY_METRIC = "roc_auc_mean"


def to_tensor(x, dtype=torch.float32):
    return torch.tensor(x, dtype=dtype, device=DEVICE)


# --------------------------------------------------------------------- #
# 1) Identifica el modelo/corrida campeon (misma logica que
#    playbook_automation.ipynb)
# --------------------------------------------------------------------- #
def select_champion():
    with open(SUMMARY_JSON_PATH, "r", encoding="utf-8") as f:
        summary_df = pd.DataFrame(json.load(f))
    all_runs_df = pd.read_csv(ALL_RUNS_PATH)

    ranked = summary_df.sort_values(
        by=[PRIMARY_METRIC, STD_TIEBREAKER, SECONDARY_METRIC],
        ascending=[False, True, False],
    ).reset_index(drop=True)
    champion_model = ranked.loc[0, "Modelo"]

    model_runs = all_runs_df[all_runs_df["model"] == champion_model]
    champion_row = model_runs.sort_values(by="f1", ascending=False).iloc[0]

    return champion_model, champion_row


# --------------------------------------------------------------------- #
# 2) Reconstruye el mismo test split que preprocessing.py, pero
#    conservando 'Dst Port' aparte (solo para agrupar, no para el modelo)
# --------------------------------------------------------------------- #
def rebuild_test_split_with_port(data):
    raw = pd.read_csv(DATA_PATH)
    raw["Label"] = raw["Label"].astype(str).str.strip()
    raw = raw.replace([np.inf, -np.inf], np.nan)

    raw_clean, _ = clean(raw, drop_leakage=False)  # conserva Dst Port
    raw_clean = raw_clean.dropna()
    raw_encoded = encode_target(raw_clean)

    dst_port_full = raw_clean["Dst Port"].values
    y_full = raw_encoded["y"].values
    idx_full = np.arange(len(y_full))

    idx_train, idx_temp, _, y_temp = train_test_split(
        idx_full, y_full, test_size=0.40, stratify=y_full, random_state=SPLIT_SEED
    )
    idx_val, idx_test, _, _ = train_test_split(
        idx_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=SPLIT_SEED
    )

    assert len(idx_test) == len(data["y_test"]), (
        "El tamano del split reconstruido no coincide con el de "
        "preprocessing.prepare_dataset(); revisa que la limpieza sea identica."
    )

    return dst_port_full[idx_test]


def port_group(p):
    if p <= 1023:
        return "well_known (0-1023)"
    elif p <= 49151:
        return "registered (1024-49151)"
    else:
        return "dynamic (49152-65535)"


# --------------------------------------------------------------------- #
# 3) Carga el modelo campeon y genera scores/predicciones sobre test
# --------------------------------------------------------------------- #
def load_champion_and_score(champion_model, champion_row, n_features):
    run_name = f"{champion_model}_seed{int(champion_row['seed'])}"
    artifact_path = f"{RESULTS_DIR}/model_{run_name}.pt"

    model = build_model(champion_model, n_features).to(DEVICE)
    model.load_state_dict(torch.load(artifact_path, map_location=DEVICE))
    model.eval()

    return model, artifact_path, run_name


def score_batch(model, champion_model, X, champion_row):
    model.eval()
    with torch.no_grad():
        if champion_model == "Autoencoder":
            X_t = to_tensor(X)
            recon = model(X_t)
            err = torch.mean((recon - X_t) ** 2, dim=1).cpu().numpy()
            score_dominant = -err
            dominant_label = int(champion_row["dominant_label_trained_on"])
            return score_dominant if dominant_label == 1 else -score_dominant
        logits = model(to_tensor(X))
        return torch.sigmoid(logits).cpu().numpy()


# --------------------------------------------------------------------- #
# 4) Metricas por grupo: ANTES (umbral global) y DESPUES (umbral por
#    grupo, F1-optimo -- mitigacion tipo post-processing)
# --------------------------------------------------------------------- #
def best_threshold_by_f1(y_true, y_score):
    thresholds = np.unique(y_score)
    if len(thresholds) > 200:
        thresholds = np.quantile(y_score, np.linspace(0, 1, 200))
    best_t, best_f1 = 0.5, -1.0
    for t in thresholds:
        pred = (y_score >= t).astype(int)
        f1 = f1_score(y_true, pred, zero_division=0)
        if f1 > best_f1:
            best_f1, best_t = f1, t
    return float(best_t)


def group_metrics(y_true, y_pred):
    fn_rate = ((y_pred == 0) & (y_true == 1)).sum() / max((y_true == 1).sum(), 1)
    return {
        "n_flujos": int(len(y_true)),
        "tasa_positiva_real": float(np.mean(y_true)),
        "tasa_positiva_predicha": float(np.mean(y_pred)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "fn_rate": float(fn_rate),
    }


def run_bias_audit():
    champion_model, champion_row = select_champion()
    print(f"Modelo campeon: {champion_model} (seed={int(champion_row['seed'])})")

    data = prepare_dataset(DATA_PATH, random_state=SPLIT_SEED)
    dst_port_test = rebuild_test_split_with_port(data)
    groups = pd.Series(dst_port_test).apply(port_group).values

    model, artifact_path, run_name = load_champion_and_score(
        champion_model, champion_row, data["n_features"]
    )
    scores = score_batch(model, champion_model, data["X_test"], champion_row)
    global_threshold = float(champion_row["threshold"])
    y_pred_global = (scores >= global_threshold).astype(int)
    y_true = data["y_test"]

    # --- ANTES: un solo umbral global para todos los grupos ---
    before = []
    for g in sorted(set(groups)):
        mask = groups == g
        if mask.sum() == 0:
            continue
        row = {"grupo": g, "umbral": global_threshold}
        row.update(group_metrics(y_true[mask], y_pred_global[mask]))
        before.append(row)
    before_df = pd.DataFrame(before)

    # --- DESPUES: umbral F1-optimo POR GRUPO (mitigacion) ---
    after = []
    for g in sorted(set(groups)):
        mask = groups == g
        if mask.sum() == 0:
            continue
        yt, sc = y_true[mask], scores[mask]
        thr = best_threshold_by_f1(yt, sc) if len(np.unique(yt)) > 1 else global_threshold
        yp = (sc >= thr).astype(int)
        row = {"grupo": g, "umbral": thr}
        row.update(group_metrics(yt, yp))
        after.append(row)
    after_df = pd.DataFrame(after)

    print("\n=== ANTES (umbral global unico =", global_threshold, ") ===")
    print(before_df.to_string(index=False))
    print("\n=== DESPUES (umbral F1-optimo por grupo) ===")
    print(after_df.to_string(index=False))

    result = {
        "modelo_campeon": champion_model,
        "corrida": run_name,
        "atributo_proxy": "Dst Port (rango de servicio)",
        "umbral_global_original": global_threshold,
        "antes": before_df.to_dict(orient="records"),
        "despues": after_df.to_dict(orient="records"),
    }

    out_path = f"{BIAS_AUDIT_DIR}/bias_audit_result.json"
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(result, f, indent=2, ensure_ascii=False)

    print("\nArchivo generado:", out_path)
    print("\n=== JSON PARA COPIAR AL CHAT ===")
    print(json.dumps(result, indent=2, ensure_ascii=False))

    return result




Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
result = run_bias_audit()

Modelo campeon: MLP (seed=43)

=== ANTES (umbral global unico = 0.9990817332387568 ) ===
                  grupo   umbral  n_flujos  tasa_positiva_real  tasa_positiva_predicha   recall  precision       f1  fn_rate
  dynamic (49152-65535) 0.999082     25373            0.000000                0.000118 0.000000   0.000000 0.000000 0.000000
registered (1024-49151) 0.999082     20636            0.000000                0.000097 0.000000   0.000000 0.000000 0.000000
    well_known (0-1023) 0.999082    162101            0.064768                0.064497 0.995619   0.999809 0.997709 0.004381

=== DESPUES (umbral F1-optimo por grupo) ===
                  grupo   umbral  n_flujos  tasa_positiva_real  tasa_positiva_predicha   recall  precision      f1  fn_rate
  dynamic (49152-65535) 0.999082     25373            0.000000                0.000118 0.000000   0.000000 0.00000 0.000000
registered (1024-49151) 0.999082     20636            0.000000                0.000097 0.000000   0.000000 0.00000 0.